# Concentration Curves

This notebook validates, combines, plots, and exports the country-level concentration curves used across multiple parts of the national tool. Curve definitions and interpretation metadata come from the explicit registry in `config/countries/KEN.toml`.

## Concentration-Curve Construction

Upstream observations are ordered by the registered ranking variable and direction. Cumulative population and cumulative outcome are each divided by their totals, explicit `(0, 0)` and `(1, 1)` endpoints are included, and the result is placed on the shared population-share grid. This notebook consumes those precomputed points; it does not silently resample them.

## 0. Setup

Shared loading and validation logic lives in `src/national_tool_metrics/concentration_curves.py`. The input and output contract is documented in `docs/concentration_curves.md`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
SRC_DIRECTORY = REPO_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from national_tool_metrics import load_country_config
from national_tool_metrics.concentration_curves import (
    SHARED_X_COLUMN,
    build_concentration_curve_output,
    concentration_curve_registry_frame,
    write_concentration_curve_output,
)

In [ ]:
config = load_country_config("MOZ", repo_root=REPO_ROOT)

print(f"Country: {config.country.name} ({config.country.iso3})")
print(f"Registered curves: {len(config.concentration_curves):,}")

## 1. Review the Registry

Check the stable curve identifiers, source columns, ranking direction, and descriptions before loading the raw CSVs. The registry order determines the output column order.

In [ ]:
curve_registry = concentration_curve_registry_frame(config)
curve_registry

## 2. Load, Validate, and Combine

Each curve must be bounded between 0 and 1, monotonic, start at `(0, 0)`, end at `(1, 1)`, and use the same cumulative-population-share grid as all other curves. The workflow deliberately does not interpolate mismatched inputs.

In [ ]:
concentration_curves = build_concentration_curve_output(config)

print(f"Rows: {len(concentration_curves):,}")
print(f"Curve columns: {len(concentration_curves.columns) - 1:,}")
display(concentration_curves.head())
display(concentration_curves.tail())

## 3. Plot and Review

The dashed diagonal is the line of equality. Registry IDs are rendered with their three components separated for readability.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
x_values = concentration_curves[SHARED_X_COLUMN]
ax.plot(x_values, x_values, linestyle="--", color="0.45", label="Line of equality")

for curve_id in config.concentration_curves:
    ax.plot(
        x_values,
        concentration_curves[curve_id],
        linewidth=2,
        label=curve_id.replace("__", " | "),
    )

ax.set(
    xlim=(0, 1),
    ylim=(0, 1),
    xlabel="Cumulative population share",
    ylabel="Cumulative outcome share",
    title=f"{config.country.name}: concentration curves",
)
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 4. Troubleshoot Concentration Indices

Calculate each concentration index directly from its plotted curve using `CI = 1 - 2 × area under the curve`. With observations ranked from lowest to highest, a negative index means the outcome is concentrated toward the lower-ranked end and a positive index means it is concentrated toward the higher-ranked end. `change_from_baseline` follows the metrics-sheet convention: adapted minus baseline.

In [ ]:
x_values = concentration_curves[SHARED_X_COLUMN].to_numpy()
area_by_curve = {}
index_by_curve = {}

for curve_id in config.concentration_curves:
    y_values = concentration_curves[curve_id].to_numpy()
    area = (
        (y_values[:-1] + y_values[1:])
        * (x_values[1:] - x_values[:-1])
    ).sum() / 2
    area_by_curve[curve_id] = area
    index_by_curve[curve_id] = 1 - (2 * area)

concentration_index_review = curve_registry[
    ["curve_id", "ranked_by", "rank_direction"]
].copy()
concentration_index_review["area_under_curve"] = (
    concentration_index_review["curve_id"].map(area_by_curve)
)
concentration_index_review["concentration_index"] = (
    concentration_index_review["curve_id"].map(index_by_curve)
)
concentration_index_review["change_from_baseline"] = float("nan")

baseline_curve_ids = [
    curve_id
    for curve_id in config.concentration_curves
    if curve_id.endswith("__baseline_protected")
]
if len(baseline_curve_ids) == 1:
    baseline_index = index_by_curve[baseline_curve_ids[0]]
    concentration_index_review["change_from_baseline"] = (
        concentration_index_review["concentration_index"] - baseline_index
    )

display(
    concentration_index_review.style.format(
        {
            "area_under_curve": "{:.6f}",
            "concentration_index": "{:.6f}",
            "change_from_baseline": "{:+.6f}",
        },
        na_rep="—",
    )
)

## 5. Export

Write one country-level CSV with the shared cumulative population share in the first column and one column per registered concentration curve.

In [ ]:
output_path = write_concentration_curve_output(concentration_curves, config)
print(f"Exported concentration curves to: {output_path}")